<div style='background: linear-gradient(135deg, #0a0a2e, #3d1278, #7F77DD); padding: 40px; border-radius: 14px; text-align: center; color: white;'>
<h1 style='font-size:2.2em; margin:0 0 8px'>PriceDetection: um novíssimo Detector de Preços</h1>
<h2 style='font-weight:400; color:#c8b8f8; margin:0 0 16px'>SQL: Window Functions, CTEs e Relatórios</h2>
<p style='color:#e0d8ff; max-width:600px; margin:0 auto; line-height:1.7'>
Com 30 dias de histórico no banco, irei explorar o poder do SQL:
<strong>window functions</strong>, <strong>CTEs encadeadas</strong>, análise de tendência
e geração automática de relatórios em CSV.

> Esta etapa foi desenvolvida com apoio de IA (ChatGPT, Claude) para geração e otimização de consultas SQL e estrutura analítica (CTEs, window functions, métricas). Todas as soluções foram revisadas, adaptadas e compreendidas por mim incluindo validação de resultados.

</p>
</div>

## Terceira etapa: O projeto

| Etapa | SQL | Motivação |
|-------|-------------|----------------|
| 1 | `RANK() / DENSE_RANK()` | Ranking de preços por produto |
| 2 | `LAG() / LEAD()` | Comparar com o dia anterior/seguinte |
| 3 | `SUM() OVER` (acumulado) | Total acumulado ao longo do tempo |
| 4 | `AVG() OVER` (média móvel) | Suavizar oscilações de preço |
| 5 | CTEs encadeadas | Queries complexas |
| 6 | `CASE WHEN` | Classificação e categorização |
| 7 | Relatório automático em CSV | Exportar análise para o Looker |

> Para esta etapa, recriaremos o banco de dados dos últimos 30 dias.

---
## Configurando o banco de dados novamente (últimos 30 dias)

In [1]:
!pip install faker pandas plotly --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 20.5 MB/s eta 0:00:00


In [5]:
import sqlite3, random, pandas as pd, plotly.express as px
import plotly.graph_objects as go
from datetime import date, timedelta
from google.colab import files

random.seed(42)
hoje = date.today()
DB_PATH = 'priceradar.db'
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cur = conn.cursor()
#conectando ao sqlite

#permite executar múltiplas instruções sql
cur.executescript('''
DROP TABLE IF EXISTS precos;
DROP TABLE IF EXISTS produtos;
DROP TABLE IF EXISTS supermercados;
DROP TABLE IF EXISTS categorias;
CREATE TABLE categorias(id INTEGER PRIMARY KEY AUTOINCREMENT, nome TEXT NOT NULL UNIQUE, descricao TEXT);
CREATE TABLE supermercados(id INTEGER PRIMARY KEY AUTOINCREMENT, nome TEXT NOT NULL UNIQUE, cidade TEXT NOT NULL, bairro TEXT, ativo INTEGER NOT NULL DEFAULT 1);
CREATE TABLE produtos(id INTEGER PRIMARY KEY AUTOINCREMENT, nome TEXT NOT NULL, marca TEXT, unidade TEXT NOT NULL, categoria_id INTEGER NOT NULL, FOREIGN KEY(categoria_id) REFERENCES categorias(id));
CREATE TABLE precos(id INTEGER PRIMARY KEY AUTOINCREMENT, produto_id INTEGER NOT NULL, supermercado_id INTEGER NOT NULL, preco REAL NOT NULL, data_coleta TEXT NOT NULL, em_promocao INTEGER NOT NULL DEFAULT 0, FOREIGN KEY(produto_id) REFERENCES produtos(id), FOREIGN KEY(supermercado_id) REFERENCES supermercados(id));
CREATE INDEX idx_precos_produto ON precos(produto_id);
CREATE INDEX idx_precos_mercado ON precos(supermercado_id);
CREATE INDEX idx_precos_data    ON precos(data_coleta);
''')
#zeramos tudo, e começamos de novo

cur.executemany('INSERT INTO categorias(nome,descricao) VALUES(?,?)',[
    ('Grãos e Cereais','Arroz, feijão, macarrão...'),('Óleos e Gorduras','Óleos, azeite...'),
    ('Laticínios','Leite, queijo, manteiga...'),('Carnes','Frango, bovina...'),
    ('Higiene Pessoal','Shampoo, sabonete...'),('Limpeza','Detergente, sabão...'),])
cur.executemany('INSERT INTO supermercados(nome,cidade,bairro) VALUES(?,?,?)',[
    ('Atacadão','Uberlândia','Brasil'),('Bretas','Uberlândia','Centro'),
    ('Superpão','Uberlândia','Santa Mônica'),('Comper','Uberlândia','Tibery'),
    ("Sam's Club",'Uberlândia','Jardim Karaíba'),])
cur.executemany('INSERT INTO produtos(nome,marca,unidade,categoria_id) VALUES(?,?,?,?)',[
    ('Arroz Branco','Tio João','5kg',1),('Arroz Branco','Camil','5kg',1),
    ('Feijão Carioca','Camil','1kg',1),('Feijão Preto','Kicaldo','1kg',1),
    ('Macarrão Espaguete','Barilla','500g',1),('Óleo de Soja','Liza','900ml',2),
    ('Azeite Extra Virgem','Galo','500ml',2),('Leite Integral','Itambé','1L',3),
    ('Queijo Mussarela','Polenghi','500g',3),('Manteiga','Aviação','200g',3),
    ('Peito de Frango','Sadia','1kg',4),('Patinho Moído','Friboi','500g',4),
    ('Shampoo','Pantene','400ml',5),('Sabonete','Dove','90g',5),
    ('Detergente','Ypê','500ml',6),('Sabão em Pó','OMO','1kg',6),])

PRECOS_BASE   = {1:22.90,2:21.50,3:8.20,4:8.90,5:6.50,6:6.10,7:28.90,
                 8:4.50,9:18.90,10:8.90,11:15.90,12:14.50,13:12.90,
                 14:2.50,15:2.50,16:10.90}
FATOR_MERCADO = {1:0.92,2:1.05,3:1.00,4:1.08,5:0.95}

#definimos os preços e os fatores e mapeamos o que cada objeto vai receber

registros = []
for d in range(29, -1, -1):
    dt = (hoje - timedelta(days=d)).isoformat()
    dias_desde = (hoje - timedelta(days=d) - date(2024,1,1)).days
    inflacao   = 1 + (dias_desde / 30) * 0.003
    for pid, base in PRECOS_BASE.items():
        for sid, fator in FATOR_MERCADO.items():
            var    = random.uniform(-0.03, 0.03)
            preco  = round(base * fator * inflacao * (1+var), 2)
            promo  = 0
            if random.random() < 0.08:
                preco = round(preco * (1 - random.uniform(0.10,0.25)), 2)
                promo = 1
            registros.append((pid, sid, preco, dt, promo))
#o loop percorre os últimos 30 dias e, para cada produto em cada supermercado, calcula um preço simulando inflação e variação aleatória diária
#em alguns casos aplica desconto (promoção) e armazena todos os registros gerados na lista registros para inserção no banco

cur.executemany('INSERT INTO precos(produto_id,supermercado_id,preco,data_coleta,em_promocao) VALUES(?,?,?,?,?)', registros)
conn.commit() #confirma inserção

total = cur.execute('SELECT COUNT(*) FROM precos').fetchone()[0] #executa a query, pega a primeira linha do resultado
print(f'Banco pronto! {total:,} registros de preço ({30} dias × {len(PRECOS_BASE)} produtos × 5 supermercados)')

Banco pronto! 2,400 registros de preço (30 dias × 16 produtos × 5 supermercados)


---
## Primeira parte: RANK() e DENSE_RANK() - Um ranking de preços

In [6]:
# Pergunta: qual posição cada supermercado ocupa no ranking de preço de hoje?

q_rank = '''
SELECT
    p.nome AS produto,
    p.unidade,
    s.nome AS supermercado,
    ROUND(pr.preco, 2) AS preco,

    RANK() OVER (
        PARTITION BY p.id  -- reinicia o ranking para cada produto
        ORDER BY pr.preco ASC  -- ordem crescente = mais barato = rank 1
    ) AS rank_preco,

    DENSE_RANK() OVER (
        PARTITION BY p.id
        ORDER BY pr.preco ASC
    ) AS dense_rank_preco,

    -- Percentil: posição relativa (0 = mais barato, 1 = mais caro)
    ROUND(
        PERCENT_RANK() OVER (
            PARTITION BY p.id
            ORDER BY pr.preco ASC
        ), 2
    ) AS percentil

FROM precos pr
JOIN produtos p ON p.id = pr.produto_id
JOIN supermercados s ON s.id = pr.supermercado_id
WHERE pr.data_coleta = DATE('now')
  AND pr.em_promocao = 0
ORDER BY p.nome, rank_preco
'''

df_rank = pd.read_sql(q_rank, conn)
print('Ranking de preços por produto (hoje, sem promoções):')
display(df_rank.head(25))

Ranking de preços por produto (hoje, sem promoções):


,produto,unidade,supermercado,preco,rank_preco,dense_rank_preco,percentil
0,Arroz Branco,5kg,Atacadão,23.37,1,1,0.00
1,Arroz Branco,5kg,Atacadão,21.04,1,1,0.00
2,Arroz Branco,5kg,Sam's Club,23.99,2,2,0.25
3,Arroz Branco,5kg,Sam's Club,21.96,2,2,0.25
4,Arroz Branco,5kg,Superpão,25.48,3,3,0.50
5,Arroz Branco,5kg,Superpão,23.41,3,3,0.50
6,Arroz Branco,5kg,Bretas,25.94,4,4,0.75
7,Arroz Branco,5kg,Bretas,24.64,4,4,0.75
8,Arroz Branco,5kg,Comper,27.52,5,5,1.00
9,Arroz Branco,5kg,Comper,24.78,5,5,1.00


In [7]:
# Heatmap: qual supermercado é mais barato para cada produto?
df_pivot = df_rank.pivot_table(
    index='produto', columns='supermercado', values='rank_preco'
).round(0)

fig = px.imshow(
    df_pivot,
    title='Mapa de ranking: rank 1 = mais barato (verde)',
    color_continuous_scale='RdYlGn_r',
    text_auto=True,
    aspect='auto'
)
fig.update_layout(
    plot_bgcolor='#0a0a2e', paper_bgcolor='#0a0a2e',
    font_color='white', title_font_size=14,
    coloraxis_colorbar=dict(title='Rank')
)
fig.show()
print('Verde = mais barato ou Vermelho = mais caro para aquele produto')

Verde = mais barato ou Vermelho = mais caro para aquele produto


---
## Segunda parte: LAG() e LEAD() - Comparando com dias diferentes

In [8]:
#LAG: acessa o valor da linha ANTERIOR
#LEAD: acessa o valor da linha SEGUINTE
#Pergunta: como o preço de cada produto variou dia a dia?

q_lag = '''
SELECT
    p.nome AS produto,
    s.nome AS supermercado,
    pr.data_coleta,
    ROUND(pr.preco, 2) AS preco_hoje,

    -- Preço do dia anterior (LAG)
    ROUND(
        LAG(pr.preco, 1) OVER (
            PARTITION BY pr.produto_id, pr.supermercado_id
            ORDER BY pr.data_coleta
        ), 2
    ) AS preco_ontem,

    -- Variação absoluta em R$
    ROUND(
        pr.preco - LAG(pr.preco, 1) OVER (
            PARTITION BY pr.produto_id, pr.supermercado_id
            ORDER BY pr.data_coleta
        ), 2
    ) AS variacao_R$,

    -- Variação percentual
    ROUND(
        (pr.preco - LAG(pr.preco, 1) OVER (
            PARTITION BY pr.produto_id, pr.supermercado_id
            ORDER BY pr.data_coleta
        )) * 100.0 / LAG(pr.preco, 1) OVER (
            PARTITION BY pr.produto_id, pr.supermercado_id
            ORDER BY pr.data_coleta
        ), 2
    ) AS variacao_pct,

    -- Preço previsto para amanhã (LEAD)
    ROUND(
        LEAD(pr.preco, 1) OVER (
            PARTITION BY pr.produto_id, pr.supermercado_id
            ORDER BY pr.data_coleta
        ), 2
    ) AS preco_amanha

FROM precos pr
JOIN produtos      p ON p.id = pr.produto_id
JOIN supermercados s ON s.id = pr.supermercado_id
WHERE p.nome  = 'Arroz Branco'
  AND p.marca = 'Tio João'
  AND s.nome  = 'Atacadão'
ORDER BY pr.data_coleta
'''
# Analisa a variação diária de preços usando window functions (LAG e LEAD)
# Compara preço atual com o anterior para calcular diferença absoluta e percentual
# Mostra também o próximo valor

df_lag = pd.read_sql(q_lag, conn)
print('Variação diária do Arroz Tio João no Atacadão:')
display(df_lag)

Variação diária do Arroz Tio João no Atacadão:


,produto,supermercado,data_coleta,preco_hoje,preco_ontem,variacao_R$,variacao_pct,preco_amanha
0,Arroz Branco,Atacadão,2026-03-26,19.73,NaN,NaN,NaN,22.28
1,Arroz Branco,Atacadão,2026-03-27,22.28,19.73,2.55,12.92,23.30
2,Arroz Branco,Atacadão,2026-03-28,23.30,22.28,1.02,4.58,22.28
3,Arroz Branco,Atacadão,2026-03-29,22.28,23.30,-1.02,-4.38,23.16
4,Arroz Branco,Atacadão,2026-03-30,23.16,22.28,0.88,3.95,22.92
5,Arroz Branco,Atacadão,2026-03-31,22.92,23.16,-0.24,-1.04,22.67
6,Arroz Branco,Atacadão,2026-04-01,22.67,22.92,-0.25,-1.09,22.49
7,Arroz Branco,Atacadão,2026-04-02,22.49,22.67,-0.18,-0.79,22.17
8,Arroz Branco,Atacadão,2026-04-03,22.17,22.49,-0.32,-1.42,22.18
9,Arroz Branco,Atacadão,2026-04-04,22.18,22.17,0.01,0.05,17.09


In [10]:
#Visualizar variação percentual diária
df_lag_clean = df_lag.dropna(subset=['variacao_pct'])

fig = go.Figure()
fig.add_bar(
    x=df_lag_clean['data_coleta'],
    y=df_lag_clean['variacao_pct'],
    marker_color=[
        '#55A868' if v >= 0 else '#C44E52'
        for v in df_lag_clean['variacao_pct']
    ],
    name='Variação %'
)
fig.add_hline(y=0, line_dash='dash', line_color='white', opacity=0.4)
fig.update_layout(
    title='Variação % diária do Arroz Tio João no Atacadão',
    xaxis_title='Data', yaxis_title='Variação (%)',
    plot_bgcolor='#0a0a2e', paper_bgcolor='#0a0a2e',
    font_color='white', title_font_size=14
)
fig.show()

---
## Terceira parte: SUM() OVER - O acumulado ao longo do tempo

In [11]:
#Qual o gasto acumulado se eu comprar leite todo dia no mesmo mercado?

q_acumulado = '''
SELECT
    s.nome AS supermercado,
    pr.data_coleta,
    ROUND(pr.preco, 2) AS preco_dia,

    -- Gasto acumulado ao longo dos 30 dias
    ROUND(
        SUM(pr.preco) OVER (
            PARTITION BY pr.supermercado_id
            ORDER BY pr.data_coleta
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ), 2
    ) AS gasto_acumulado,

    -- Quantas compras foram feitas até hoje
    ROW_NUMBER() OVER (
        PARTITION BY pr.supermercado_id
        ORDER BY pr.data_coleta
    ) AS compra_numero

FROM precos pr
JOIN supermercados s ON s.id = pr.supermercado_id
JOIN produtos      p ON p.id = pr.produto_id
WHERE p.nome      = 'Leite Integral'
  AND pr.em_promocao = 0
ORDER BY pr.data_coleta, s.nome
'''

df_acum = pd.read_sql(q_acumulado, conn)
print('Gasto acumulado comprando Leite Integral todo dia:')
display(df_acum.tail(10))

#Quanto cada mercado custa no total dos 30 dias?
resumo_acum = df_acum.groupby('supermercado')['gasto_acumulado'].max().sort_values()
print('\nTotal gasto em 30 dias por supermercado:')
for mercado, total in resumo_acum.items():
    print(f'  {mercado:<15}: R$ {total:.2f}')
economia = resumo_acum.iloc[-1] - resumo_acum.iloc[0]
print(f'\nDiferença entre mais barato e mais caro: R$ {economia:.2f}')

Gasto acumulado comprando Leite Integral todo dia:


,supermercado,data_coleta,preco_dia,gasto_acumulado,compra_numero
131,Superpão,2026-04-22,4.76,131.24,27
132,Atacadão,2026-04-23,4.50,125.56,28
133,Bretas,2026-04-23,5.12,149.15,29
134,Comper,2026-04-23,5.12,136.19,26
135,Superpão,2026-04-23,4.77,136.01,28
136,Atacadão,2026-04-24,4.47,130.03,29
137,Bretas,2026-04-24,4.99,154.14,30
138,Comper,2026-04-24,5.25,141.44,27
139,Sam's Club,2026-04-24,4.53,120.24,26
140,Superpão,2026-04-24,4.78,140.79,29



Total gasto em 30 dias por supermercado:
  Sam's Club     : R$ 120.24
  Atacadão       : R$ 130.03
  Superpão       : R$ 140.79
  Comper         : R$ 141.44
  Bretas         : R$ 154.14

Diferença entre mais barato e mais caro: R$ 33.90


In [13]:
fig = px.line(
    df_acum, x='data_coleta', y='gasto_acumulado', color='supermercado',
    title='Gasto acumulado do Leite Integral',
    labels={'data_coleta':'Data','gasto_acumulado':'Gasto acumulado (R$)'},
    markers=True
)
fig.update_layout(
    plot_bgcolor='#0a0a2e', paper_bgcolor='#0a0a2e',
    font_color='white', title_font_size=14
)
fig.show()

---
## Quarta parte: AVG() OVER - Média Móvel

In [15]:
#Média móvel
#Calcula médias móveis (3 e 7 dias) para suavizar o preço diário
#A de 3 dias reage rápido; a de 7 dias mostra a tendência mais estável
#Permite comparar com o preço real para identificar oscilações vs tendência

q_mm = '''
SELECT
    s.nome AS supermercado,
    pr.data_coleta,
    ROUND(pr.preco, 2) AS preco_real,

    -- Média móvel de 3 dias (janela menor, mais sensível)
    ROUND(
        AVG(pr.preco) OVER (
            PARTITION BY pr.supermercado_id
            ORDER BY pr.data_coleta
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW   -- 3 dias: anteontem, ontem, hoje
        ), 2
    ) AS media_movel_3d,

    -- Média móvel de 7 dias (janela maior, mais suave)
    ROUND(
        AVG(pr.preco) OVER (
            PARTITION BY pr.supermercado_id
            ORDER BY pr.data_coleta
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW   -- 7 dias
        ), 2
    ) AS media_movel_7d

FROM precos pr
JOIN supermercados s ON s.id = pr.supermercado_id
JOIN produtos      p ON p.id = pr.produto_id
WHERE p.nome      = 'Peito de Frango'
  AND s.nome      = 'Bretas'
ORDER BY pr.data_coleta
'''

df_mm = pd.read_sql(q_mm, conn)
print('Média móvel do Peito de Frango no Bretas:')
display(df_mm)

Média móvel do Peito de Frango no Bretas:


,supermercado,data_coleta,preco_real,media_movel_3d,media_movel_7d
0,Bretas,2026-03-26,18.01,18.01,18.01
1,Bretas,2026-03-27,18.53,18.27,18.27
2,Bretas,2026-03-28,18.39,18.31,18.31
3,Bretas,2026-03-29,18.27,18.40,18.30
4,Bretas,2026-03-30,17.65,18.10,18.17
5,Bretas,2026-03-31,18.11,18.01,18.16
6,Bretas,2026-04-01,18.13,17.96,18.16
7,Bretas,2026-04-02,18.50,18.25,18.23
8,Bretas,2026-04-03,17.54,18.06,18.08
9,Bretas,2026-04-04,17.86,17.97,18.01


In [16]:
fig = go.Figure()
fig.add_scatter(x=df_mm['data_coleta'], y=df_mm['preco_real'],
    mode='markers', name='Preço real', marker=dict(color='#A89CF7', size=5, opacity=0.6))
fig.add_scatter(x=df_mm['data_coleta'], y=df_mm['media_movel_3d'],
    mode='lines', name='Média móvel 3d', line=dict(color='#55A868', width=2))
fig.add_scatter(x=df_mm['data_coleta'], y=df_mm['media_movel_7d'],
    mode='lines', name='Média móvel 7d', line=dict(color='#DD8452', width=2.5))

fig.update_layout(
    title='Preço real vs. médias móveis do Peito de Frango (Bretas)',
    xaxis_title='Data', yaxis_title='Preço (R$)',
    plot_bgcolor='#0a0a2e', paper_bgcolor='#0a0a2e',
    font_color='white', title_font_size=14
)
fig.show()
print('Média móvel de 7 dias é mais suave.')

Média móvel de 7 dias é mais suave.


---
## Quinta parte: CTEs Encadeadas

In [17]:
#CTEs(Common Table Expressions)

# Cada CTE é uma parte da análise, responde uma pergunta complexa
# Quais produtos tiveram tendência de ALTA de preço e ainda assim têm o maior potencial de economia entre mercados?
# O problema foi quebrado em 4 etapas: o preço semanal, a variação semanal, a tendência e a economia em potencial.

q_cte = '''
-- CTE 1: preço médio por produto por semana
WITH preco_semanal AS (
    SELECT
        produto_id,
        STRFTIME('%W', data_coleta)   AS semana,
        ROUND(AVG(preco), 2) AS media_semana
    FROM precos
    WHERE em_promocao = 0 -- filtra apenas produtos sem promoção
    GROUP BY produto_id, semana
),

-- CTE 2: variação entre semanas (usando LAG dentro da CTE)
variacao_semanal AS (
    SELECT
        produto_id,
        semana,
        media_semana,
        LAG(media_semana) OVER (
            PARTITION BY produto_id
            ORDER BY semana
        ) AS media_semana_anterior,
        ROUND(
            (media_semana - LAG(media_semana) OVER ( -- calcula a variação em relação ao anterior
                PARTITION BY produto_id ORDER BY semana -- divide pela anterior
            )) * 100.0 / LAG(media_semana) OVER (
                PARTITION BY produto_id ORDER BY semana
            ), 2
        ) AS variacao_pct
    FROM preco_semanal
),

-- CTE 3: tendência geral (positiva = alta, negativa = queda)
tendencia AS (
    SELECT
        produto_id,
        ROUND(AVG(variacao_pct), 2)   AS tendencia_pct,
        CASE
            WHEN AVG(variacao_pct) >  0.5 THEN 'Alta'
            WHEN AVG(variacao_pct) < -0.5 THEN 'Queda'
            ELSE 'Estável'
        END
    AS classificacao
    FROM variacao_semanal
    WHERE variacao_pct IS NOT NULL
    GROUP BY produto_id
),

-- CTE 4: potencial de economia (diferença max-min entre mercados hoje)
economia_potencial AS (
    SELECT
        produto_id,
        ROUND(MAX(preco) - MIN(preco), 2) AS economia_R$,
        ROUND((MAX(preco)-MIN(preco))*100.0/MIN(preco), 1) AS economia_pct
    FROM precos
    WHERE data_coleta = DATE('now') AND em_promocao = 0
    GROUP BY produto_id
)

-- Query final: combina as 4 CTEs
SELECT
    p.nome AS produto,
    p.unidade,
    t.classificacao AS tendencia_preco,
    t.tendencia_pct AS variacao_media_semanal_pct,
    e.economia_R$,
    e.economia_pct,
    CASE
        WHEN t.classificacao = 'Alta' AND e.economia_pct > 10
            THEN 'Atenção: subindo e vale pesquisar preço'
        WHEN t.classificacao = 'Queda' AND e.economia_pct > 10
            THEN 'Ótimo: caindo e tem economia entre mercados'
        WHEN e.economia_pct > 15
            THEN 'Sempre compare: diferença grande entre mercados'
        ELSE 'Preço estável e diferença pequena'
    END                    AS recomendacao
FROM tendencia t
JOIN economia_potencial e ON e.produto_id = t.produto_id
JOIN produtos            p ON p.id        = t.produto_id
ORDER BY e.economia_pct DESC
'''

df_cte = pd.read_sql(q_cte, conn)
print('Análise: tendência de preço + potencial de economia:')
display(df_cte)

Análise: tendência de preço + potencial de economia:


,produto,unidade,tendencia_preco,variacao_media_semanal_pct,economia_R$,economia_pct,recomendacao
0,Patinho Moído,500g,Estável,0.11,3.10,21.9,Sempre compare: diferença grande entre mercados
1,Peito de Frango,1kg,Estável,-0.20,3.17,20.1,Sempre compare: diferença grande entre mercados
2,Manteiga,200g,Estável,0.00,1.63,18.6,Sempre compare: diferença grande entre mercados
3,Sabonete,90g,Estável,-0.19,0.46,18.5,Sempre compare: diferença grande entre mercados
4,Arroz Branco,5kg,Estável,0.27,4.15,17.8,Sempre compare: diferença grande entre mercados
5,Arroz Branco,5kg,Estável,0.01,3.74,17.8,Sempre compare: diferença grande entre mercados
6,Feijão Preto,1kg,Estável,-0.03,1.53,17.7,Sempre compare: diferença grande entre mercados
7,Shampoo,400ml,Estável,-0.02,2.21,17.6,Sempre compare: diferença grande entre mercados
8,Feijão Carioca,1kg,Estável,-0.11,1.43,17.5,Sempre compare: diferença grande entre mercados
9,Leite Integral,1L,Estável,-0.10,0.78,17.4,Sempre compare: diferença grande entre mercados


---
## Sexta parte: CASE WHEN - Classificação e categorização

In [19]:
#Como classificar o nível de preço de cada produto hoje?

q_case = '''
WITH stats_produto AS (
    -- Calcula média e desvio padrão histórico de cada produto
    SELECT
        produto_id,
        AVG(preco) AS media_hist,
        -- Desvio padrão manual (SQLite não tem STDDEV nativo)
        SQRT(AVG(preco * preco) - AVG(preco) * AVG(preco)) AS desvio_hist
    FROM precos
    WHERE em_promocao = 0
    GROUP BY produto_id
)
SELECT
    p.nome AS produto,
    s.nome AS supermercado,
    ROUND(pr.preco, 2) AS preco_hoje,
    ROUND(sp.media_hist, 2) AS media_historica,
    ROUND(sp.desvio_hist, 2) AS desvio,

    -- Classificação baseada em quantos desvios padrão o preço está da média
    CASE
        WHEN pr.preco < sp.media_hist - sp.desvio_hist THEN '🟢 Muito barato'
        WHEN pr.preco < sp.media_hist THEN '🔵 Abaixo da média'
        WHEN pr.preco < sp.media_hist + sp.desvio_hist THEN '🟡 Acima da média'
        ELSE '🔴 Caro'
    END AS classificacao_preco,

    -- Variação em relação à média histórica
    ROUND(
        (pr.preco - sp.media_hist) * 100.0 / sp.media_hist,
    1) AS vs_media_pct

FROM precos pr
JOIN produtos p ON p.id  = pr.produto_id
JOIN supermercados s  ON s.id  = pr.supermercado_id
JOIN stats_produto sp ON sp.produto_id = pr.produto_id
WHERE pr.data_coleta = DATE('now')
  AND pr.em_promocao = 0
ORDER BY vs_media_pct ASC
'''

df_case = pd.read_sql(q_case, conn)
print('Classificação de preço vs. histórico:')
display(df_case.head(20))

print('\nDistribuição das classificações:')
print(df_case['classificacao_preco'].value_counts())

Classificação de preço vs. histórico:


,produto,supermercado,preco_hoje,media_historica,desvio,classificacao_preco,vs_media_pct
0,Feijão Preto,Atacadão,8.62,9.61,0.62,🟢 Muito barato,-10.3
1,Patinho Moído,Atacadão,14.16,15.74,1.00,🟢 Muito barato,-10.0
2,Arroz Branco,Atacadão,21.04,23.35,1.50,🟢 Muito barato,-9.9
3,Shampoo,Atacadão,12.59,13.96,0.87,🟢 Muito barato,-9.8
4,Manteiga,Atacadão,8.75,9.65,0.62,🟢 Muito barato,-9.3
5,Peito de Frango,Atacadão,15.75,17.26,1.04,🟢 Muito barato,-8.8
6,Leite Integral,Atacadão,4.47,4.87,0.30,🟢 Muito barato,-8.2
7,Macarrão Espaguete,Atacadão,6.48,7.05,0.43,🟢 Muito barato,-8.1
8,Feijão Carioca,Atacadão,8.17,8.88,0.55,🟢 Muito barato,-8.0
9,Sabonete,Atacadão,2.49,2.71,0.16,🟢 Muito barato,-8.0



Distribuição das classificações:
classificacao_preco
🟡 Acima da média     23
🔵 Abaixo da média    20
🟢 Muito barato       18
🔴 Caro               13
Name: count, dtype: int64


In [20]:
# Desvio padrão (STDDEV - calculado manualmente no SQLite):
# Mede o quanto o preço varia em relação à média.
# Fórmula usada: sqrt(média dos quadrados - (média)^2)
# Quanto maior o desvio, mais instável é o preço do produto.

# CASE (classificação de preço):
# Classifica o preço atual comparando com média e desvio padrão.

# vs_media_pct:
# Mostra em porcentagem o quanto o preço atual está acima ou abaixo da média histórica (ajuda a comparar produtos diferentes em escala relativa)

---
## Sétima parte: Relatório Automático em CSV

In [21]:
#Relatório 1: Melhores preços do dia
q_relatorio_dia = '''
WITH ranking AS (
    SELECT
        p.nome AS produto,
        p.marca,
        p.unidade,
        c.nome AS categoria,
        s.nome AS supermercado,
        s.bairro,
        ROUND(pr.preco, 2) AS preco,
        pr.data_coleta,
        RANK() OVER (
            PARTITION BY p.id
            ORDER BY pr.preco ASC
        ) AS rank_preco
    FROM precos pr
    JOIN produtos p ON p.id = pr.produto_id
    JOIN supermercados s ON s.id = pr.supermercado_id
    JOIN categorias c ON c.id = p.categoria_id
    WHERE pr.data_coleta = DATE('now') AND pr.em_promocao = 0
)
SELECT * FROM ranking
ORDER BY categoria, produto, rank_preco
'''
df_rel1 = pd.read_sql(q_relatorio_dia, conn)
df_rel1.to_csv('relatorio_precos_hoje.csv', index=False, encoding='utf-8-sig')
print(f'relatorio_precos_hoje.csv — {len(df_rel1)} linhas')
display(df_rel1.head(10))

relatorio_precos_hoje.csv — 74 linhas


,produto,marca,unidade,categoria,supermercado,bairro,preco,data_coleta,rank_preco
0,Patinho Moído,Friboi,500g,Carnes,Atacadão,Brasil,14.16,2026-04-24,1
1,Patinho Moído,Friboi,500g,Carnes,Sam's Club,Jardim Karaíba,15.05,2026-04-24,2
2,Patinho Moído,Friboi,500g,Carnes,Superpão,Santa Mônica,15.65,2026-04-24,3
3,Patinho Moído,Friboi,500g,Carnes,Bretas,Centro,16.19,2026-04-24,4
4,Patinho Moído,Friboi,500g,Carnes,Comper,Tibery,17.26,2026-04-24,5
5,Peito de Frango,Sadia,1kg,Carnes,Atacadão,Brasil,15.75,2026-04-24,1
6,Peito de Frango,Sadia,1kg,Carnes,Sam's Club,Jardim Karaíba,16.44,2026-04-24,2
7,Peito de Frango,Sadia,1kg,Carnes,Superpão,Santa Mônica,17.12,2026-04-24,3
8,Peito de Frango,Sadia,1kg,Carnes,Bretas,Centro,18.12,2026-04-24,4
9,Peito de Frango,Sadia,1kg,Carnes,Comper,Tibery,18.92,2026-04-24,5


In [22]:
#Relatório 2: Histórico completo com média móvel
q_historico = '''
SELECT
    p.nome AS produto,
    p.marca,
    p.unidade,
    c.nome AS categoria,
    s.nome AS supermercado,
    pr.data_coleta,
    ROUND(pr.preco, 2) AS preco,
    pr.em_promocao,
    ROUND(
        AVG(pr.preco) OVER (
            PARTITION BY pr.produto_id, pr.supermercado_id
            ORDER BY pr.data_coleta
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ), 2
    ) AS media_movel_7d,
    ROUND(
        pr.preco - LAG(pr.preco) OVER (
            PARTITION BY pr.produto_id, pr.supermercado_id
            ORDER BY pr.data_coleta
        ), 2
    ) AS variacao_dia_R$
FROM precos pr
JOIN produtos p ON p.id = pr.produto_id
JOIN supermercados s ON s.id = pr.supermercado_id
JOIN categorias c ON c.id = p.categoria_id
ORDER BY p.nome, s.nome, pr.data_coleta
'''
df_rel2 = pd.read_sql(q_historico, conn)
df_rel2.to_csv('relatorio_historico_completo.csv', index=False, encoding='utf-8-sig')
print(f'relatorio_historico_completo.csv — {len(df_rel2):,} linhas')
display(df_rel2.head(8))

relatorio_historico_completo.csv — 2,400 linhas


,produto,marca,unidade,categoria,supermercado,data_coleta,preco,em_promocao,media_movel_7d,variacao_dia_R$
0,Arroz Branco,Tio João,5kg,Grãos e Cereais,Atacadão,2026-03-26,19.73,1,19.73,NaN
1,Arroz Branco,Camil,5kg,Grãos e Cereais,Atacadão,2026-03-26,18.62,1,18.62,NaN
2,Arroz Branco,Tio João,5kg,Grãos e Cereais,Atacadão,2026-03-27,22.28,0,21.01,2.55
3,Arroz Branco,Camil,5kg,Grãos e Cereais,Atacadão,2026-03-27,21.35,0,19.99,2.73
4,Arroz Branco,Tio João,5kg,Grãos e Cereais,Atacadão,2026-03-28,23.30,0,21.77,1.02
5,Arroz Branco,Camil,5kg,Grãos e Cereais,Atacadão,2026-03-28,21.18,0,20.38,-0.17
6,Arroz Branco,Tio João,5kg,Grãos e Cereais,Atacadão,2026-03-29,22.28,0,21.90,-1.02
7,Arroz Branco,Camil,5kg,Grãos e Cereais,Atacadão,2026-03-29,21.03,0,20.55,-0.15


In [23]:
#Relatório 3: Resumo por supermercado
q_resumo = '''
WITH base AS (
    SELECT
        s.nome AS supermercado,
        c.nome AS categoria,
        COUNT(DISTINCT pr.produto_id) AS produtos,
        ROUND(AVG(pr.preco), 2) AS preco_medio,
        ROUND(MIN(pr.preco), 2) AS menor_preco,
        ROUND(MAX(pr.preco), 2) AS maior_preco,
        SUM(pr.em_promocao) AS qtd_promocoes,
        COUNT(*) AS total_coletas,
        ROUND(SUM(pr.em_promocao)*100.0/COUNT(*), 1) AS pct_promocoes
    FROM precos pr
    JOIN supermercados s ON s.id = pr.supermercado_id
    JOIN categorias c ON c.id = (
        SELECT categoria_id FROM produtos WHERE id = pr.produto_id
    )
    GROUP BY s.id, c.id
)
SELECT *,
    RANK() OVER (PARTITION BY categoria ORDER BY preco_medio ASC) AS rank_categoria
FROM base
ORDER BY categoria, rank_categoria
'''
df_rel3 = pd.read_sql(q_resumo, conn)
df_rel3.to_csv('relatorio_resumo_executivo.csv', index=False, encoding='utf-8-sig')
print(f'relatorio_resumo_executivo.csv — {len(df_rel3)} linhas')
display(df_rel3)

relatorio_resumo_executivo.csv — 30 linhas


,supermercado,categoria,produtos,preco_medio,menor_preco,maior_preco,qtd_promocoes,total_coletas,pct_promocoes,rank_categoria
0,Atacadão,Carnes,2,15.03,12.74,16.32,3,60,5.0,1
1,Sam's Club,Carnes,2,15.49,11.53,16.83,4,60,6.7,2
2,Superpão,Carnes,2,16.34,11.81,17.70,3,60,5.0,3
3,Bretas,Carnes,2,17.12,12.85,18.62,3,60,5.0,4
4,Comper,Carnes,2,17.61,12.99,19.13,4,60,6.7,5
5,Atacadão,Grãos e Cereais,5,13.27,5.14,23.51,16,150,10.7,1
6,Sam's Club,Grãos e Cereais,5,13.86,5.24,24.25,8,150,5.3,2
7,Superpão,Grãos e Cereais,5,14.54,5.35,25.50,10,150,6.7,3
8,Bretas,Grãos e Cereais,5,15.28,5.56,26.71,12,150,8.0,4
9,Comper,Grãos e Cereais,5,15.82,7.10,27.52,5,150,3.3,5


In [24]:
#Baixar tudo
conn.commit()
conn.close()

print('Baixando arquivos...\n')
for arq in ['priceradar.db',
            'relatorio_precos_hoje.csv',
            'relatorio_historico_completo.csv',
            'relatorio_resumo_executivo.csv']:
    files.download(arq)
    print(f' {arq}')

Baixando arquivos...



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 priceradar.db


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

 relatorio_historico_completo.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 relatorio_resumo_executivo.csv
